In [7]:
!pip install psycopg2-binary python-dotenv reportlab statsmodels scikit-learn -q

In [ ]:
import os
import time
import psutil
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from datetime import datetime

from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier


# =========================
# 1. CARREGAR .ENV
# =========================

load_dotenv(".env")


# =========================
# 2. CONEXÃO COM O BANCO
# =========================

def conectar_banco():
    return psycopg2.connect(
        user=os.getenv("DB_USER"),
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT"),
        sslmode="require"
    )


# =========================
# 3. NOVA TABELA PARA INSERÇÃO DE DADOS
# =========================

def criar_tabela_monitoramento():
    conn = conectar_banco()
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS monitoramento_recursos (
            id SERIAL PRIMARY KEY,
            data_hora TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            cpu_percent NUMERIC,
            memoria_percent NUMERIC,
            disco_percent NUMERIC,
            processos INTEGER
        );
    """)

    conn.commit()
    cursor.close()
    conn.close()

    print("Tabela monitoramento_recursos pronta.")


# =========================
# 4. COLETAR E SALVAR DADOS
# =========================

def coletar_e_salvar_metricas(qtd_amostras=30, intervalo=2):
    conn = conectar_banco()
    cursor = conn.cursor()

    print("Coletando métricas da instância...")

    for i in range(qtd_amostras):
        cpu = psutil.cpu_percent(interval=1)
        memoria = psutil.virtual_memory().percent
        disco = psutil.disk_usage("/").percent
        processos = len(psutil.pids())

        cursor.execute("""
            INSERT INTO monitoramento_recursos
            (cpu_percent, memoria_percent, disco_percent, processos)
            VALUES (%s, %s, %s, %s);
        """, (cpu, memoria, disco, processos))

        conn.commit()

        print(f"Amostra {i + 1}/{qtd_amostras} salva | CPU: {cpu}% | Memória: {memoria}% | Disco: {disco}%")

        time.sleep(intervalo)

    cursor.close()
    conn.close()

    print("Coleta finalizada.")


# =========================
# 5. BUSCAR DADOS DO BANCO
# =========================

def buscar_dados():
    conn = conectar_banco()

    query = """
        SELECT
            id,
            data_hora,
            cpu_percent AS cpu,
            memoria_percent AS memoria,
            disco_percent AS disco,
            processos
        FROM monitoramento_recursos
        ORDER BY id ASC;
    """

    df = pd.read_sql_query(query, conn)
    conn.close()

    df["cpu"] = pd.to_numeric(df["cpu"])
    df["memoria"] = pd.to_numeric(df["memoria"])
    df["disco"] = pd.to_numeric(df["disco"])
    df["processos"] = pd.to_numeric(df["processos"])
    df["tempo_seq"] = np.arange(len(df))

    return df


# =========================
# 6. MODELOS DE IA
# =========================

def aplicar_regressao_linear(df):
    X = df[["tempo_seq"]]
    y = df["cpu"]

    modelo = LinearRegression()
    modelo.fit(X, y)

    proximos_tempos = np.array([[len(df)], [len(df) + 1], [len(df) + 2], [len(df) + 3], [len(df) + 4]])
    predicoes = modelo.predict(proximos_tempos)

    return predicoes


def aplicar_kmeans(df):
    X = df[["cpu", "memoria", "disco"]]

    modelo = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters = modelo.fit_predict(X)

    return clusters


def aplicar_arvore_decisao(df):
    df = df.copy()

    df["anomalia"] = (
        (df["cpu"] > 75) |
        (df["memoria"] > 80) |
        (df["disco"] > 90)
    ).astype(int)

    X = df[["cpu", "memoria", "disco", "processos"]]
    y = df["anomalia"]

    modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
    modelo.fit(X, y)

    predicoes = modelo.predict(X)

    return predicoes


# =========================
# 7. GERAR GRÁFICOS
# =========================

def gerar_graficos(df):
    pred_cpu = aplicar_regressao_linear(df)
    clusters = aplicar_kmeans(df)
    anomalias = aplicar_arvore_decisao(df)

    df["cluster"] = clusters
    df["anomalia"] = anomalias

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("Monitoramento de Recursos da Instância com IA", fontsize=16, fontweight="bold")

    # Gráfico 1 - CPU, Memória e Disco
    axes[0, 0].plot(df["data_hora"], df["cpu"], marker="o", label="CPU %")
    axes[0, 0].plot(df["data_hora"], df["memoria"], marker="o", label="Memória %")
    axes[0, 0].plot(df["data_hora"], df["disco"], marker="o", label="Disco %")
    axes[0, 0].set_title("Uso de Recursos ao Longo do Tempo")
    axes[0, 0].set_xlabel("Data/Hora")
    axes[0, 0].set_ylabel("Percentual de Uso")
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Gráfico 2 - Predição de CPU
    eixo_real = df["tempo_seq"]
    eixo_pred = range(len(df), len(df) + 5)

    axes[0, 1].plot(eixo_real, df["cpu"], marker="o", label="CPU real")
    axes[0, 1].plot(eixo_pred, pred_cpu, marker="x", linestyle="--", label="Predição CPU")
    axes[0, 1].set_title("Regressão Linear - Predição de CPU")
    axes[0, 1].set_xlabel("Tempo sequencial")
    axes[0, 1].set_ylabel("CPU %")
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Gráfico 3 - Clusters
    scatter = axes[1, 0].scatter(df["cpu"], df["memoria"], c=df["cluster"], s=80)
    axes[1, 0].set_title("K-Means - Agrupamento de Padrões")
    axes[1, 0].set_xlabel("CPU %")
    axes[1, 0].set_ylabel("Memória %")
    axes[1, 0].grid(True)

    # Gráfico 4 - Anomalias
    cores = ["red" if x == 1 else "blue" for x in df["anomalia"]]

    axes[1, 1].scatter(df["data_hora"], df["cpu"], c=cores, s=80)
    axes[1, 1].set_title("Árvore de Decisão - Detecção de Anomalias")
    axes[1, 1].set_xlabel("Data/Hora")
    axes[1, 1].set_ylabel("CPU %")
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig("graficos_monitoramento_ia.png", dpi=300)
    plt.show()

    print("Gráfico salvo como: graficos_monitoramento_ia.png")

    return df, pred_cpu


# =========================
# 8. RELATÓRIO SIMPLES
# =========================

def gerar_relatorio_simples(df, pred_cpu):
    total_registros = len(df)
    media_cpu = df["cpu"].mean()
    media_memoria = df["memoria"].mean()
    media_disco = df["disco"].mean()
    total_anomalias = df["anomalia"].sum()

    texto = f"""
RELATÓRIO DE MONITORAMENTO E ANÁLISE DE DESEMPENHO

Objetivo:
Este relatório apresenta o monitoramento de recursos de uma instância em nuvem,
considerando CPU, memória, disco e quantidade de processos.

Dados analisados:
- Total de registros coletados: {total_registros}
- Média de CPU: {media_cpu:.2f}%
- Média de memória: {media_memoria:.2f}%
- Média de disco: {media_disco:.2f}%
- Total de anomalias detectadas: {total_anomalias}

Modelos de IA aplicados:

1. Regressão Linear:
Utilizada para prever os próximos valores de uso de CPU com base no histórico coletado.

Próximas predições de CPU:
{[round(x, 2) for x in pred_cpu]}

2. K-Means:
Utilizado para agrupar os registros em padrões de comportamento parecidos,
considerando CPU, memória e disco.

3. Árvore de Decisão:
Utilizada para classificar possíveis anomalias.
Foi considerada anomalia quando CPU > 75%, memória > 80% ou disco > 90%.

Conclusão:
A implementação permitiu coletar dados reais da instância, armazená-los no banco PostgreSQL,
aplicar modelos simples de Inteligência Artificial e gerar gráficos de monitoramento.
"""

    with open("relatorio_monitoramento_ia.txt", "w", encoding="utf-8") as arquivo:
        arquivo.write(texto)

    print("Relatório salvo como: relatorio_monitoramento_ia.txt")


# =========================
# 9. EXECUÇÃO
# =========================

criar_tabela_monitoramento()

coletar_e_salvar_metricas(
    qtd_amostras=30,
    intervalo=2
)

df = buscar_dados()

if len(df) < 3:
    raise Exception("É necessário ter pelo menos 3 registros para aplicar os modelos.")

df_resultado, pred_cpu = gerar_graficos(df)

gerar_relatorio_simples(df_resultado, pred_cpu)

In [9]:
from google.colab import files

files.download("graficos_monitoramento_ia.png")
files.download("relatorio_monitoramento_ia.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>